In [1]:
from pathlib import Path

from lassi.profiler import Timer, MultiProfiler, GPUProfiler, CPUProfiler, Report
from lassi.source_file import SourceFile
from lassi.executer import FunctionalValidator
from typing import Annotated

from lassi.compiler import Compiler, CompilerTool, CompilationError

from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

import subprocess

import dotenv
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai.models.groq import GroqModel

from openai import AsyncOpenAI

In [2]:
source_file = SourceFile(file_name = Path("test.c"))
profiler = MultiProfiler([GPUProfiler(), CPUProfiler()])

In [12]:
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

dotenv.load_dotenv()

#MODEL_NAME = "openai/gpt-oss-120b"
#model = GroqModel(MODEL_NAME)

model = GoogleModel('gemini-3-flash-preview')
agent = Agent(model, tools=[duckduckgo_search_tool()])

In [13]:
output = await agent.run(
    'Use the search tool and tell me how the chicago bulls did this season'
    )

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from pydantic_graph import BaseNode, End, Graph, GraphRunContext

@dataclass
class LASSIState:
    original_file : SourceFile
    original_report : Report = None
    best_file : SourceFile = None
    edited_files : list[SourceFile] = field(default_factory=list)
    edited_files_reports : list[Report] = field(default_factory=list)
    compilation_message: str | None = None
    execution_message: str | None = None
    refactoring_plan: str | None = None
    i : int = 0
    iterations: int = 3

@dataclass
class EditorAgentDeps:
    source_file: SourceFile
    plan : str | None = None

@dataclass
class FileStatus(BaseModel):
    file_name: str
    content: str

@dataclass
class Plan(BaseNode[LASSIState]):
    def __init__(self):
        model = GoogleModel('gemini-3-pro-preview')
        self.agent = Agent(
            model=model,
            instructions=(
                "You are an expert C coder helping to optimize code for performance. Your goal is to generate a step-by-step refactoring plan for the provided C code."
                "The DuckDuckGo search tool is available for any information you might need."
            ),
            tools=[duckduckgo_search_tool()],
            deps_type=EditorAgentDeps,
        )

        super().__init__()

    async def run(self, ctx: GraphRunContext) -> Compile:
        
        print("Entering Planning Node")

        ctx.state.i += 1
        # set the best file if it's the first iteration
        if ctx.state.best_file is None:
            ctx.state.best_file = ctx.state.original_file.copy(file_name=f"best_{ctx.state.original_file.file_name}")

        # set current edited file
        ctx.state.edited_files.append(ctx.state.best_file.copy(file_name=f"edited_{ctx.state.i}_{ctx.state.original_file.file_name}"))

        print(ctx.state)

        if ctx.state.compilation_message:
            # If there was a compilation error, ask the agent to fix it
            refactoring_plan = await self.agent.run(
                user_prompt = f'The following compilation error occurred: [ERRORS]{ctx.state.compilation_message}[\ERRORS]\nIdentify the issues in the code and propose a step-by-step plan to fix them.[CODE]{ctx.state.edited_files[-1].read()}[/CODE]',
                deps = EditorAgentDeps(source_file=ctx.state.edited_files[-1])
                )
        else:
            refactoring_plan = await self.agent.run(
                user_prompt = f'Identify the optimization opportunities in the code and propose a step-by-step plan to optimize it.\n[CODE]{ctx.state.edited_files[-1].read()}[/CODE]',
                deps = EditorAgentDeps(source_file=ctx.state.edited_files[-1])
                )
        
        ctx.state.refactoring_plan = refactoring_plan.output

        return Edit()

@dataclass
class Edit(BaseNode[LASSIState]):
    def __init__(self):
        model = GoogleModel('gemini-3-flash-preview')
        self.agent = Agent(
            model=model,
            deps_type=EditorAgentDeps,
        )

        @self.agent.tool
        async def write(
            ctx: RunContext[EditorAgentDeps], optimized_code: str
        ) -> None:
            """Write the content of the source file."""
            ctx.deps.source_file.write(optimized_code)

        @self.agent.system_prompt
        def add_context(ctx: RunContext[EditorAgentDeps]) -> str:
            return (
                f"You are a C optimization expert.\n"
                f"Use this plan to guide your edits."
                f"REFACTORING PLAN: {ctx.deps.plan}\n"
            )

        super().__init__()

    async def run(self, ctx: GraphRunContext) -> Compile:
        
        print("Entering Edit Node")

        print(ctx.state)

        await self.agent.run(
            user_prompt=f"Optimize the following code:\n{ctx.state.edited_files[-1].read()}",
            deps=EditorAgentDeps(
                source_file=ctx.state.edited_files[-1], 
                plan=ctx.state.refactoring_plan
            )
        )

        return Eval()

class EvalAgentDeps(BaseModel):
    original_file: SourceFile
    refactored_file: SourceFile
    plan: str | None = None

class EvalAgentOutput(BaseModel):
    functional_equivalence:  Annotated[bool, Field(description="Whether the optimized code is functionally equivalent to the original code.")]
    functional_equivalence_comment: Annotated[str, Field(description="Comment on the functional equivalence of the optimized code to the original code.")]
    adherence_to_plan: Annotated[bool, Field(description="Whether the optimized code adheres to the provided refactoring plan.")]
    adherence_to_plan_comment: Annotated[str, Field(description="Comment on the adherence of the optimized code to the provided refactoring plan.")]

@dataclass
class Eval(BaseNode[LASSIState]):
    def __init__(self):
        model = GoogleModel('gemini-3-pro-preview')
        self.agent = Agent(
            model=model,
            instructions=(
                "You are an expert C coder helping to optimize code for performance. Your goal is to analyze if the code has been optimized according to the provided plan."
                "The DuckDuckGo search tool is available for any information you might need."
            ),
            tools=[duckduckgo_search_tool()],
            deps_type=EditorAgentDeps,
        )

        super().__init__()

    async def run(self, ctx: GraphRunContext) -> Compile:
        
        print("Entering Eval Node")

        print(ctx.state)

        self.agent.run(
            user_prompt=f"""Analyze if the following code has been optimized according to the provided plan
                            [ORIGINAL CODE]{ctx.state.original_file.read()}[\CODE]
                            [OPTIMIZED CODE]{ctx.state.edited_files[-1].read()}[\CODE]
                            [REFACTORING PLAN]{ctx.state.refactoring_plan}[\PLAN]""")

        return Compile()

@dataclass
class Compile(BaseNode[LASSIState]):
    async def run(self, ctx: GraphRunContext) -> Execute | Edit:
        """Compile the source file."""

        print("Entering Compile Node")
        print(ctx.state)

        try: 
            ctx.state.edited_files[-1].compile()
            ctx.state.compilation_message = None
            return Execute()
        
        except Exception as e:
            ctx.state.compilation_message = f"Compilation failed: {str(e)}"
            ctx.state.edited_files_reports.append(None)
            return Edit()

@dataclass
class Execute(BaseNode[LASSIState, None, int]):
    async def run(self, ctx: GraphRunContext) -> Edit | End[int]:
        """Execute the compiled source file with profiling."""

        print("Entering Execute Node")
        print(ctx.state)

        try:
            source_file = ctx.state.edited_files[-1] if ctx.state.i > 0 else ctx.state.original_file

            report = source_file.execute(profiler=profiler)

            print(f"Execution successful. Report: {report}")
            ctx.state.execution_message = None
            if ctx.state.i > 0:
                ctx.state.edited_files_reports.append(report)
            else:
                ctx.state.original_report = report

            if ctx.state.i < ctx.state.iterations:
                return Edit()
            else:
                return End()
        except Exception as e:
            if ctx.state.i < ctx.state.iterations:
                return Edit()
            else:
                return Failure()

@dataclass
class Failure(BaseNode[LASSIState, None, int]):
    async def run(self, ctx: GraphRunContext) -> Edit | End[int]:
        """Failure node."""
        print("Entering Failure Node")
        print(ctx.state)
        return End("Optimization process terminated due to repeated failures.")

lassi_graph = Graph(nodes=(Edit, Compile, Execute,Failure))

<>:61: SyntaxWarning: invalid escape sequence '\E'
<>:61: SyntaxWarning: invalid escape sequence '\E'
/tmp/ipykernel_2320836/1954581010.py:61: SyntaxWarning: invalid escape sequence '\E'
  user_prompt = f'The following compilation error occurred: [ERRORS]{ctx.state.compilation_message}[\ERRORS]\nIdentify the issues in the code and propose a step-by-step plan to fix them.[CODE]{ctx.state.edited_files[-1].read()}[/CODE]',


In [18]:
print(lassi_graph.mermaid_code(start_node=Execute))

NameError: name 'lassi_graph' is not defined

In [ ]:
output = await lassi_graph.run(
    start_node=Execute(),
    state=LASSIState(original_file=source_file),
)